# 02 · Klasifikasi Hujan — Bab 3

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 3: membangun model klasifikasi biner (hujan/tidak hujan) dan mempelajari precision/recall/F1 serta trade-off threshold.

Prasyarat: Bab 1 (`ch-01-00_fondasi_tensorflow`) dan Bab 2 (`ch-02-01_regresi_pasang_surut`).

## 1. Setup & Verifikasi Lingkungan

In [1]:
import tensorflow as tf
import numpy as np

print("TensorFlow:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0


GPU tersedia: []


## 2. Data Sintetik Tidak Seimbang

Kita buat data sederhana meniru prediksi hujan deras (>50 mm) yang jarang terjadi: mayoritas 'tidak hujan deras', beberapa 'hujan deras'.

In [2]:
np.random.seed(42)

n = 2000
# Fitur: [kelembapan, tekanan] ~ dua kelompok
x = np.random.randn(n, 2)
prob_deras = 0.05  # hanya 5% hujan deras
y = (np.random.rand(n) < (prob_deras + 0.4 * (x[:,0] > 1))).astype(float)

print("Distribusi label:")
print("  Hujan deras:", int(y.sum()), "| Tidak hujan deras:", int((1-y).sum()))

Distribusi label:
  Hujan deras: 225 | Tidak hujan deras: 1775


## 3. Split Berbasis Waktu

Meski sintetik, kita tetap memakai split urutan (bukan acak) untuk konsistensi dengan data deret waktu nyata.

In [3]:
n_train = int(n * 0.7)
n_val = int(n * 0.15)

X_train, y_train = x[:n_train], y[:n_train]
X_val, y_val = x[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test, y_test = x[n_train+n_val:], y[n_train+n_val:]
print(f"train {X_train.shape} | val {X_val.shape} | test {X_test.shape}")

train (1400, 2) | val (300, 2) | test (300, 2)


## 4. Model Klasifikasi Biner (Naif, tanpa bobot)

Arsitektur Kode 3.1: lapisan keluaran **sigmoid** dan loss `binary_crossentropy`. Model ini adalah *baseline naif* — tanpa menangani ketidakseimbangan. Kita akan lihat bahwa di threshold 0.5 ia gagalnya total pada kejadian langka: demostrasi jebakan akurasi tinggi di Bagian 3.6.

In [4]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(8, activation="relu", input_shape=(2,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()],
)
model.summary()

C:\Users\Hi\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105 (420.00 B)

 Trainable params: 105 (420.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=32, verbose=0,
)
print("Metric terakhir (train/val):")
for k in history.history:
    print(f"  {k}: {history.history[k][-1]:.4f}")

Metric terakhir (train/val):
  accuracy: 0.8879
  loss: 0.2606
  precision: 0.4516
  recall: 0.1854
  val_accuracy: 0.8867
  val_loss: 0.2807
  val_precision: 0.5385
  val_recall: 0.2000


## 5. Akurasi vs Metrik: Jebakan Model Naif

Tekst Bagian 3.6: akurasi tinggi bisa menipu. Kita hitung confusion matrix pada beberapa threshold (0.2, 0.5, 0.8) untuk model naif. Anticipasi: threshold 0.5 ke atas TP = 0 — model berpikir "tidak hujan deras" karena kelas langka sangat minoritas.

In [6]:
def hitung_metrik(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    prec = tp / (tp + fp) if (tp + fp) else 0
    rec = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0
    acc = (tp + tn) / (tp + fp + fn + tn)
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": round(prec,3), "recall": round(rec,3),
            "F1": round(f1,3), "akurasi": round(acc,3)}

y_prob = model.predict(X_test, verbose=0).ravel()
print("Model naif (tanpa bobot):")
for thr in [0.2, 0.5, 0.8]:
    print(f"  Threshold {thr}: {hitung_metrik(y_test, y_prob, thr)}")

Model naif (tanpa bobot):
  Threshold 0.2: {'TP': 25, 'FP': 32, 'FN': 14, 'TN': 229, 'precision': 0.439, 'recall': 0.641, 'F1': 0.521, 'akurasi': 0.847}
  Threshold 0.5: {'TP': 7, 'FP': 11, 'FN': 32, 'TN': 250, 'precision': 0.389, 'recall': 0.179, 'F1': 0.246, 'akurasi': 0.857}
  Threshold 0.8: {'TP': 0, 'FP': 0, 'FN': 39, 'TN': 261, 'precision': 0, 'recall': 0.0, 'F1': 0, 'akurasi': 0.87}


## 6. Solusi: Bobot Kelas (Class Weight) — Model Utama

Teks Kode 3.3: `class_weight` memberi penalti lebih besar untuk kesalahan pada kelas minoritas. Melatih model **bobot kelas** — ini model utama untuk evaluasi lanjut. Bandingkan hasil dengan model naif di Bagian 8.

In [7]:
class_weight = {0: 1.0, 1: 10.0}

model_w = tf.keras.Sequential([
    tf.keras.layers.Dense(8, activation="relu", input_shape=(2,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model_w.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()],
)

history_w = model_w.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    class_weight=class_weight,
    epochs=50, batch_size=32, verbose=0,
)

## 7. Trade-off Threshold dengan Model Utama

Model utama (dengan bobot) menunjukkan trade-off yang jelas: threshold rendah → lebih banyak kejadian tertangkap (recall naik); threshold tinggi → lebih sedikit false alarm (precision naik). Ini diskusi Bagian 3.7.

In [8]:
y_prob_w = model_w.predict(X_test, verbose=0).ravel()
print("Model utama (bobot kelas 10):")
for thr in [0.2, 0.5, 0.8]:
    print(f"  Threshold {thr}: {hitung_metrik(y_test, y_prob_w, thr)}")

Model utama (bobot kelas 10):
  Threshold 0.2: {'TP': 39, 'FP': 261, 'FN': 0, 'TN': 0, 'precision': 0.13, 'recall': 1.0, 'F1': 0.23, 'akurasi': 0.13}
  Threshold 0.5: {'TP': 27, 'FP': 52, 'FN': 12, 'TN': 209, 'precision': 0.342, 'recall': 0.692, 'F1': 0.458, 'akurasi': 0.787}
  Threshold 0.8: {'TP': 12, 'FP': 20, 'FN': 27, 'TN': 241, 'precision': 0.375, 'recall': 0.308, 'F1': 0.338, 'akurasi': 0.843}


## 8. Bandingkan: Senza vs Con Bobot Kelas

Bandingkan di threshold 0.5: model naif gagalnya total (recall 0) sedangkan model bobot menangkap kejadian deras. Ini perbandingan yang disarankan di teks Kode 3.3.

In [9]:
print("Threshold 0.5 - tanpa bobot:", hitung_metrik(y_test, y_prob, 0.5))
print("Threshold 0.5 - bobot 10   :", hitung_metrik(y_test, y_prob_w, 0.5))

Threshold 0.5 - tanpa bobot: {'TP': 7, 'FP': 11, 'FN': 32, 'TN': 250, 'precision': 0.389, 'recall': 0.179, 'F1': 0.246, 'akurasi': 0.857}
Threshold 0.5 - bobot 10   : {'TP': 27, 'FP': 52, 'FN': 12, 'TN': 209, 'precision': 0.342, 'recall': 0.692, 'F1': 0.458, 'akurasi': 0.787}


## 9. Diskusi

- **Trade-off threshold**: threshold 0.2 → recall lebih tinggi tetapi precision turun (false alarm lebih banyak); threshold 0.8 → precision naik tetapi kejadian terlewat. Tidak ada jawaban universal — tergantung biaya (Bagian 3.7).
- **Model naif gagalnya di 0.5**: akurasi tetap tinggi (~0.87) karena mayoritas 'tidak hujan deras' — persis jebakan Bagian 3.6: akurasi bukan metrik yang tepat untuk fenomena langka.
- **Bobot kelas menangkap kejadian langka**: di threshold 0.5, recall model bobot jauh lebih tinggi daripada model naif.
- Bab 5 akan memperkenalkan CSI/FAR/POD/TS sesuai pedoman WMO; Bab 9 menerapkannya pada data stasiun BMKG.

## 10. Latihan Mini

1. Ubah `prob_deras` menjadi 0.5 (seimbang) — bandingkan metriknya.
2. Latih model multi-kelas (3 kelas intensitas) dan lihat confusion matrix.
3. Plot precision-recall untuk beberapa threshold.
4. Diskusikan: threshold mana yang Anda pilih jika false alarm mahal? Jika miss mahal?